In [13]:
#%pip install SimilarityText <--- uncomment to install SimilarityText in Jupyter Notebook

In [14]:
from similarity import Similarity
from pathlib import Path
import pandas as pd

In [15]:
# initialise directories
base_dir = Path("./Data/comment_sections/tagged/posttag")

subfolders = [
    "ausunions",
    "onenationoz",
    "weareunion",
    "paulinehansononenation",
]

proportions_file = Path("./Output/posttag_tag_proportions.csv")

output_file = Path("./Output/posttag_tag_proportions_similarity.csv")

In [16]:
# initialise similarity model
sim = Similarity(
    language="english",
    langdetect=False,
    quiet=True
)

In [18]:
# create results array
results = []

# loop through subfolder
for subfolder in subfolders:

    folder = base_dir / subfolder

    csv_files = sorted(folder.glob("*.csv"))

    # loop through files in each subfolder
    for csv_file in csv_files:

        df = pd.read_csv(csv_file)

        if "comment" not in df.columns:
            print("    Skipping: no 'comment' column", flush=True)
            continue

        # clean 'comments'
        comments = (
            df["comment"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        # remove empty comments
        comments = comments[comments != ""].tolist()

        n = len(comments)

        # similarity results array
        similarity_scores = []
        
        # pairwise similarity comparison
        for i in range(n):

            for j in range(i + 1, n):

                score = sim.similarity(
                    comments[i],
                    comments[j]
                )

                similarity_scores.append(score)

        # calculate mean similarity across all pairs
        mean_similarity = sum(similarity_scores) / len(similarity_scores)

        # append results to results array
        results.append({
            "subfolder": subfolder,
            "post_id": csv_file.stem,
            "text_similarity_score": mean_similarity
        })


# create results data frame
results_df = pd.DataFrame(
    results,
    columns=[
        "subfolder",
        "post_id",
        "text_similarity_score"
    ]
)

# 'finished statement' to verify completion - pairwise similarity may take a while (approx 2 mins)
print("\nFinished.", flush=True)


Finished.


In [20]:
# read existing proportions csv into a df
proportions_df = pd.read_csv(proportions_file)


proportions_df["post_id"] = proportions_df["post_id"].astype(str)
results_df["post_id"] = results_df["post_id"].astype(str)


# append similarity score to proportions_df on subfolder/post_id
merged_df = proportions_df.merge(
    results_df[
        [
            "subfolder",
            "post_id",
            "text_similarity_score"
        ]
    ],
    on=["subfolder", "post_id"],
    how="left"
)


# verify
print(
    "Similarity scores missing:",
    merged_df["text_similarity_score"].isna().sum()
)

print("\nPreview:")
print(merged_df.head())


# save to csv
merged_df.to_csv(
    output_file,
    index=False
)

print("\nSaved to:", output_file)

Similarity scores missing: 0

Preview:
   subfolder post_id             video_id  play_count  pro_phonp  anti_phonp  \
0  ausunions       9  7654497847743614225      262000   0.791667    0.166667   
1  ausunions       6  7657853389975358721      179000   0.260870    0.434783   
2  ausunions       7  7657481781075070225      138000   0.548387    0.322581   
3  ausunions       8  7655116065130794261       75700   0.057143    0.800000   
4  ausunions       5  7665275616815631636        8386   0.341463    0.487805   

    unclear  n_comments                                               file  \
0  0.041667          48  Data\comment_sections\tagged\posttag\ausunions...   
1  0.304348          46  Data\comment_sections\tagged\posttag\ausunions...   
2  0.129032          31  Data\comment_sections\tagged\posttag\ausunions...   
3  0.142857          35  Data\comment_sections\tagged\posttag\ausunions...   
4  0.170732          41  Data\comment_sections\tagged\posttag\ausunions...   

   text_sim